# 갑상선 결절 초음파 분할 (TN3K) 불균형 세그멘테이션 실험

---

## 1. 태스크 및 도메인
- **도메인**: 갑상선 결절 (Thyroid Nodule) 초음파 세그멘테이션
- **모달리티**: Ultrasound (초음파 그레이스케일 이미지)
- **태스크**: Binary segmentation — 배경(0) / 결절(1)
- **핵심 도전**: 초음파 특유의 speckle 노이즈, 불명확한 병변 경계, 결절-주변 조직 유사 에코

## 2. 모델
- **아키텍처**: U-Net (ResNet34 백본)
- **사전학습**: ImageNet pretrained
- **선택 이유**: Binary segmentation 표준 베이스라인, 다도메인 비교 일관성
- **출력**: 1채널 sigmoid → `to_2ch_logits()` 변환 후 손실 함수 적용

## 3. 데이터셋
- **이름**: TN3K (Thyroid Nodule 3K)
- **규모**: 3,493장 — trainval fold JSON으로 Train/Val 분리, 공식 test set 별도 제공
- **입력 해상도**: 256×256 (리사이즈)
- **클래스 불균형**: BG:Nodule = **~20:1** (중간 수준)
- **공식 분할**: fold JSON (`tn3k-trainval-fold0.json`) + 별도 test-image/test-mask

## 4. 데이터 준비 (협업자용)
> Google Drive에 폴더를 업로드한 후 노트북을 실행할 것. Cell 0 실행 시 자동으로 /tmp 에 복사.

**취득 방법**:
- GitHub: https://github.com/haifangong/TRFE-Net-for-thyroid-nodule-segmentation

**Colab Drive 업로드 경로**:
```
MyDrive/imbalanced-data-LWCE/tn3k/
  tg3k/
    thyroid-image/*.jpg
    thyroid-mask/*.jpg
  tn3k/
    trainval-image/*.jpg
    trainval-mask/*.jpg
    test-image/*.jpg
    test-mask/*.jpg
    tn3k-trainval-fold0.json
```

## 5. 전처리 및 도메인 특이점
- BGR→RGB 변환, 마스크 이진화(임계값 128), ImageNet 정규화
- 증강: HorizontalFlip, VerticalFlip, RandomRotate90, RandomResizedCrop(size=(256,256))
- 초음파 이미지: 그레이스케일이나 3채널 BGR로 저장됨 → RGB 변환 필수

## 6. 실험 손실 함수 및 Optuna 탐색 범위
| 손실 함수 | 탐색 파라미터 | 탐색 범위 | Trials |
|-----------|-------------|----------|--------|
| `ce_dice` | — | — | — |
| `wce_dice` | — | — | — |
| `lwce_dice` | — | — | — |
| `plwce_dice` | alpha | 2.5 ~ 15.0 | 20 |
| `pwce_dice` | alpha | 0.2 ~ 2.5 | 20 |
| `cb_dice` | — | — | — |
| `plwce_focal_dice` | alpha + gamma | alpha 2.5~15.0, gamma 0.5~5.0 | 40 |

## 7. SoTA 참고 (2025년 1월 기준)
| 방법 | Dice | IoU | Sensitivity | AUC | 출처 |
|------|------|-----|-------------|-----|------|
| MRDB (Mamba+ResNet, 2024) | **90.02%** | **81.85%** | 89.11% | — | PMC/Bioengineering |
| ResUNet (2025) | 84.24% | 75.48% | 88.98% | — | AIMS Medical Science |
| DPAM-UNet++ (2024) | 83.10% | 74.51% | 87.02% | 0.9213 | BMC Medical Imaging |
| TRFE-Net/TRFE+ (2021/2022) | ~83% | ~71% | — | — | ISBI'21/CBM'22 |
| U-Net (ResNet34) baseline | ~78~82% | — | — | — | 복수 논문 |

> 본 연구 목표: U-Net baseline 대비 LWCE/PLWCE 계열 손실 함수의 개선 효과 검증.
> 평가 지표: Dice, Sensitivity, Specificity, AUC
> 결과 저장: `medical_data/results/TN3K_Thyroid_Ultrasound/`

In [ ]:
# === Cell 0: 환경 설정 ===
import subprocess, sys

for pkg in ['segmentation-models-pytorch', 'optuna', 'openpyxl', 'albumentations']:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

import os, warnings, json, random, glob, shutil
warnings.filterwarnings('ignore')

import numpy as np
import cv2
import pandas as pd
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
import albumentations as A
from albumentations.pytorch import ToTensorV2
import segmentation_models_pytorch as smp
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

# --- custom_losses 경로 ---
_CL_LOCAL = '/root/imbalanced-data-LWCE/medical_data'
_CL_COLAB = '/tmp/custom_losses'
if os.path.exists(_CL_LOCAL):
    sys.path.insert(0, _CL_LOCAL)
else:
    sys.path.insert(0, _CL_COLAB)
from custom_losses import get_loss_function

# --- Google Drive 마운트 ---
GDRIVE_DATA_PATH = '/content/drive/MyDrive/imbalanced-data-LWCE/tn3k'

IS_COLAB = False
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IS_COLAB = True
    _cl_src = '/content/drive/MyDrive/imbalanced-data-LWCE/medical_data/custom_losses.py'
    if not os.path.exists(os.path.join(_CL_LOCAL, 'custom_losses.py')) and os.path.exists(_cl_src):
        os.makedirs(_CL_COLAB, exist_ok=True)
        shutil.copy(_cl_src, _CL_COLAB)
    print('Google Drive 마운트 완료')
except Exception:
    print('Colab 환경 아님 — 로컬/수동 경로 사용')

# --- 실험 설정 ---
DOMAIN      = 'tn3k'
NUM_CLASSES = 2
CLASS_NAMES = ['Background', 'Nodule']
IMG_SIZE    = 256
BATCH_SIZE  = 16
NUM_WORKERS = 4
SEED        = 42

MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
STD  = np.array([0.229, 0.224, 0.225], dtype=np.float32)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

RESULTS_DIR = '/root/imbalanced-data-LWCE/medical_data/results/TN3K_Thyroid_Ultrasound'
os.makedirs(RESULTS_DIR, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
print('환경 설정 완료')

In [ ]:
# === Cell 1: 데이터 로드 ===
#
# [데이터 준비 방법]
# --- Google Colab + Google Drive (필수) ---
#   1. TN3K 데이터를 다운로드 (GitHub: haifangong/TRFE-Net 또는 공식 배포처)
#   2. Google Drive에 폴더 업로드:
#      MyDrive/imbalanced-data-LWCE/tn3k/
#        tg3k/thyroid-image/*.jpg
#        tg3k/thyroid-mask/*.jpg
#        tn3k/trainval-image/*.jpg  +  tn3k/trainval-mask/*.jpg
#        tn3k/test-image/*.jpg      +  tn3k/test-mask/*.jpg
#        tn3k/tn3k-trainval-fold0.json
#   3. Cell 0 실행 → 자동으로 /tmp/tn3k_data/ 에 복사
# -------------------------------------------------------------

TMP_BASE = '/tmp/tn3k_data'

# Google Drive → /tmp 복사 (Colab 환경)
if IS_COLAB and os.path.exists(GDRIVE_DATA_PATH):
    img_check = glob.glob(os.path.join(TMP_BASE, 'tg3k', 'thyroid-image', '*.jpg'))
    if len(img_check) < 100:
        print('Google Drive에서 데이터 복사 중... (최초 1회)')
        os.makedirs(TMP_BASE, exist_ok=True)
        shutil.copytree(GDRIVE_DATA_PATH, TMP_BASE, dirs_exist_ok=True)
        img_check = glob.glob(os.path.join(TMP_BASE, 'tg3k', 'thyroid-image', '*.jpg'))
        print(f'복사 완료: 이미지 {len(img_check)}개')
    else:
        print(f'캐시 사용: {len(img_check)}개 이미지 이미 존재')
    BASE_DIR = TMP_BASE
else:
    raise RuntimeError(
        'Google Drive 마운트가 필요합니다.\n'
        '  1. Cell 0 재실행 (Drive 마운트 확인)\n'
        '  2. Drive 경로 확인: ' + GDRIVE_DATA_PATH
    )

TN3K_TRAIN_IMG  = os.path.join(BASE_DIR, 'tn3k', 'trainval-image')
TN3K_TRAIN_MASK = os.path.join(BASE_DIR, 'tn3k', 'trainval-mask')
TEST_IMG  = os.path.join(BASE_DIR, 'tn3k', 'test-image')
TEST_MASK = os.path.join(BASE_DIR, 'tn3k', 'test-mask')
FOLD_JSON = os.path.join(BASE_DIR, 'tn3k', 'tn3k-trainval-fold0.json')
FOLD_NUM  = 0   # 0~4 중 선택

# --- fold JSON으로 train / val 분할 ---
with open(FOLD_JSON) as f:
    fold = json.load(f)

def idx_to_path(idx, img_dir, mask_dir):
    fname = f'{idx:04d}.jpg'
    return os.path.join(img_dir, fname), os.path.join(mask_dir, fname)

tr_imgs,  tr_masks  = [], []
val_imgs, val_masks = [], []

for idx in fold['train']:
    ip, mp = idx_to_path(idx, TN3K_TRAIN_IMG, TN3K_TRAIN_MASK)
    if os.path.exists(ip) and os.path.exists(mp):
        tr_imgs.append(ip);  tr_masks.append(mp)

for idx in fold['val']:
    ip, mp = idx_to_path(idx, TN3K_TRAIN_IMG, TN3K_TRAIN_MASK)
    if os.path.exists(ip) and os.path.exists(mp):
        val_imgs.append(ip); val_masks.append(mp)

test_imgs  = sorted(glob.glob(os.path.join(TEST_IMG,  '*.jpg')))
test_masks = sorted(glob.glob(os.path.join(TEST_MASK, '*.jpg')))

print(f'Fold {FOLD_NUM}  →  Train: {len(tr_imgs)} | Val: {len(val_imgs)} | Test: {len(test_imgs)}')
assert len(tr_imgs) > 0,  'Train 이미지를 찾지 못했습니다. BASE_DIR 경로를 확인하세요.'

# --- Augmentation 파이프라인 ---
# [Domain shift 대응]
#   - train/val 이미지: tn3k/train-image (TN3K 결절 마스크, 가변 크기)
#   - test  이미지: tn3k/test-image   (TN3K 결절 마스크, 가변 크기)
#   LongestMaxSize + PadIfNeeded 로 종횡비 보존 리사이즈
#   RandomBrightnessContrast + GaussNoise 로 스캐너 간 intensity 차이 대응
#   ElasticTransform + GridDistortion 으로 해부학적 변형 대응

_MEAN = MEAN.tolist()
_STD  = STD.tolist()

train_tf = A.Compose([
    # [핵심] 스케일 다양성 — tn3k train/test 모두 가변 크기, 작은 결절부터 큰 결절까지 학습
    # scale=(0.05, 1.0): 전체의 5%~100% 영역을 무작위 crop → 작은 결절(0.3% FG)부터
    #                    큰 결절(63% FG)까지 모든 스케일 학습
    A.LongestMaxSize(max_size=IMG_SIZE * 3),
    A.PadIfNeeded(min_height=IMG_SIZE * 3, min_width=IMG_SIZE * 3,
                  border_mode=0, value=0, mask_value=0),
    A.RandomResizedCrop(size=(IMG_SIZE, IMG_SIZE),
                        scale=(0.05, 1.0), ratio=(0.5, 2.0)),
    # 기하학적 augmentation
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.15, rotate_limit=15,
                       border_mode=0, p=0.5),
    A.ElasticTransform(alpha=60, sigma=6, p=0.3),
    A.GridDistortion(num_steps=5, distort_limit=0.3, p=0.2),
    # intensity augmentation — 초음파 스캐너 간 차이 모사
    A.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.3, p=0.7),
    A.GaussNoise(var_limit=(10.0, 50.0), p=0.5),
    A.GaussianBlur(blur_limit=(3, 5), p=0.3),
    # dropout — 프로브 아티팩트/그림자 모사
    A.CoarseDropout(max_holes=4, max_height=32, max_width=32, p=0.3),
    A.Normalize(mean=_MEAN, std=_STD),
    ToTensorV2(),
])

val_tf = A.Compose([
    # 종횡비 보존 리사이즈 (test 이미지 크기 다양성 대응 핵심)
    A.LongestMaxSize(max_size=IMG_SIZE),
    A.PadIfNeeded(min_height=IMG_SIZE, min_width=IMG_SIZE,
                  border_mode=0, value=0, mask_value=0),
    A.Normalize(mean=_MEAN, std=_STD),
    ToTensorV2(),
])

# --- Dataset 클래스 ---
class TN3KDataset(Dataset):
    """
    TN3K Thyroid Nodule Ultrasound Dataset.
    Label: 0=Background, 1=Nodule
    Input: 3-channel RGB, albumentations pipeline 적용
    """
    def __init__(self, img_paths, mask_paths, transform=None):
        self.img_paths  = img_paths
        self.mask_paths = mask_paths
        self.transform  = transform

    def __len__(self):
        return len(self.img_paths)

    def __getitem__(self, idx):
        img  = cv2.imread(self.img_paths[idx])
        mask = cv2.imread(self.mask_paths[idx], cv2.IMREAD_GRAYSCALE)

        if img is None:
            img  = np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)
        if mask is None:
            mask = np.zeros((IMG_SIZE, IMG_SIZE), dtype=np.uint8)

        img  = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        mask = (mask > 128).astype(np.uint8)   # uint8 for albumentations

        if self.transform:
            aug  = self.transform(image=img, mask=mask)
            img  = aug['image']          # float32 tensor (C, H, W)
            mask = aug['mask'].long()    # int64 tensor (H, W)
        else:
            img  = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
            img  = (img.astype(np.float32) / 255.0 - MEAN) / STD
            img  = torch.from_numpy(img.transpose(2, 0, 1).astype(np.float32))
            mask = torch.from_numpy(mask.astype(np.int64))

        return img, mask

# --- DataLoader ---
train_loader = DataLoader(
    TN3KDataset(tr_imgs,   tr_masks,   transform=train_tf),
    batch_size=BATCH_SIZE, shuffle=True,  num_workers=NUM_WORKERS, pin_memory=True
)
val_loader = DataLoader(
    TN3KDataset(val_imgs,  val_masks,  transform=val_tf),
    batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True
)
test_loader = DataLoader(
    TN3KDataset(test_imgs, test_masks, transform=val_tf),
    batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True
)
print('DataLoader 구성 완료')

In [ ]:
# === Cell 2: 클래스 비율 계산 ===
print('클래스 비율 계산 중 (학습 이미지 픽셀 단위)...')

class_counts = np.zeros(NUM_CLASSES, dtype=np.int64)
for mp in tqdm(tr_masks, desc='Counting pixels'):
    mask = cv2.imread(mp, cv2.IMREAD_GRAYSCALE)
    if mask is None:
        continue
    mask = cv2.resize(mask, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_NEAREST)
    binary = (mask > 128).astype(np.int64)
    class_counts[0] += int((binary == 0).sum())
    class_counts[1] += int((binary == 1).sum())

class_counts = class_counts.tolist()
total = sum(class_counts)

print()
for c, (name, cnt) in enumerate(zip(CLASS_NAMES, class_counts)):
    print(f'  [{c}] {name:<12}: {cnt:>15,} pixels  ({100 * cnt / total:.2f}%)')

ratio = class_counts[0] / class_counts[1]
print(f'\nBG : Nodule = {ratio:.1f} : 1')
print(f'\nclass_counts = {class_counts}')

In [ ]:
# === Cell 3: 모델 정의 ===
#
# [SoTA 참고]
#   TRFE-Net (Gong et al., 2021): Dice ≈ 0.821, IoU ≈ 0.760
#   U-Net (ResNet34):             Dice ≈ 0.78~0.82
#   출처: TN3K paper / TRFE-Net
#
# [선택 이유]
#   - 손실 함수 효과 분리 측정 → 경량 U-Net(ResNet34) 고정
#   - smp.Unet(ResNet34, ImageNet) = 재현 가능한 공개 구현
#
# [Binary seg 핵심]
#   - 모델 출력: 1채널 logit (B, 1, H, W)
#   - 손실 계산: to_2ch_logits(logit) → (B, 2, H, W) → CrossEntropyLoss
#   - 추론: sigmoid(logit) → 확률맵 → 0.5 임계값 → 예측

def to_2ch_logits(p):
    """1채널 logit → 2채널 logit (rules.md §6-2)"""
    return torch.cat([-p, p], dim=1)


def build_model():
    """U-Net (ResNet34, ImageNet pretrained) — Binary thyroid nodule segmentation."""
    return smp.Unet(
        encoder_name    = 'resnet34',
        encoder_weights = 'imagenet',
        in_channels     = 3,
        classes         = 1,       # 1채널 logit 출력
        activation      = None,
    ).to(device)


def compute_val_dice(model, loader):
    """빠른 Val Dice — Optuna 및 학습 모니터링용"""
    model.eval()
    tp = fp = fn = 0
    with torch.no_grad():
        for imgs, masks in loader:
            imgs, masks = imgs.to(device), masks.to(device)
            prob = torch.sigmoid(model(imgs)[:, 0])
            pred = (prob > 0.5).long()
            tp += ((pred == 1) & (masks == 1)).sum().item()
            fp += ((pred == 1) & (masks == 0)).sum().item()
            fn += ((pred == 0) & (masks == 1)).sum().item()
    return float(2 * tp / (2 * tp + fp + fn + 1e-8))


def compute_val_metrics(model, loader):
    """전체 Val 지표: Dice, Sensitivity, Specificity, AUC"""
    model.eval()
    all_probs, all_preds, all_labels = [], [], []
    with torch.no_grad():
        for imgs, masks in loader:
            imgs = imgs.to(device)
            prob = torch.sigmoid(model(imgs)[:, 0]).cpu().numpy()  # (B, H, W)
            pred = (prob > 0.5).astype(np.int64)
            all_probs.append(prob.flatten())
            all_preds.append(pred.flatten())
            all_labels.append(masks.numpy().flatten())

    all_probs  = np.concatenate(all_probs)
    all_preds  = np.concatenate(all_preds)
    all_labels = np.concatenate(all_labels)

    TP = ((all_preds == 1) & (all_labels == 1)).sum()
    FP = ((all_preds == 1) & (all_labels == 0)).sum()
    TN = ((all_preds == 0) & (all_labels == 0)).sum()
    FN = ((all_preds == 0) & (all_labels == 1)).sum()

    dice = 2 * TP / (2 * TP + FP + FN + 1e-8)
    sens = TP / (TP + FN + 1e-8)
    spec = TN / (TN + FP + 1e-8)
    try:
        auc = roc_auc_score(all_labels, all_probs)
    except Exception:
        auc = 0.0

    return {'Dice': float(dice), 'Sensitivity': float(sens),
            'Specificity': float(spec), 'AUC': float(auc)}


print('모델 + 유틸리티 함수 정의 완료 (U-Net ResNet34, ~21.8M params)')

In [ ]:
# === Cell 4: 학습 함수 ===

def train_model(
    loss_name,
    alpha=1.0,
    gamma=2.0,
    epochs=50,
    lr=1e-4,
    subset_ratio=1.0,
    tag='',
):
    """
    U-Net(ResNet34) 학습 함수.
    Binary seg: 1채널 logit → to_2ch_logits → get_loss_function
    """
    model     = build_model()
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    criterion = get_loss_function(loss_name, class_counts=class_counts, alpha=alpha, gamma=gamma)

    name = f'{loss_name}_alpha{alpha:.2f}' if alpha != 1.0 else loss_name
    if tag:
        name = f'{tag}_{name}'

    print(f"\n{'='*60}\nU-Net(ResNet34) + {name}  (epochs={epochs})\n{'='*60}")

    if subset_ratio < 1.0:
        n = max(1, int(len(train_loader.dataset) * subset_ratio))
        sub_ds = torch.utils.data.Subset(
            train_loader.dataset,
            random.sample(range(len(train_loader.dataset)), n)
        )
        loader = DataLoader(sub_ds, batch_size=BATCH_SIZE,
                            shuffle=True, num_workers=NUM_WORKERS)
    else:
        loader = train_loader

    history    = {'loss': [], 'val_dice': []}
    best_dice  = 0.0
    save_path  = f'/tmp/best_unet_{name}.pth'

    for epoch in range(epochs):
        model.train()
        epoch_loss = 0.0

        for imgs, masks in tqdm(loader, desc=f'Ep{epoch+1:02d}/{epochs}', leave=False, disable=os.environ.get('TQDM_DISABLE') == '1'):
            imgs, masks = imgs.to(device), masks.to(device)
            optimizer.zero_grad()
            # 1채널 logit → 2채널 logit (rules.md §6-2)
            logits_2ch = to_2ch_logits(model(imgs))
            loss = criterion(logits_2ch, masks)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()

        scheduler.step()
        avg_loss = epoch_loss / len(loader)
        val_dice = compute_val_dice(model, val_loader)

        history['loss'].append(avg_loss)
        history['val_dice'].append(val_dice)

        print(f'Ep{epoch+1:02d} | Loss: {avg_loss:.4f} | Val Dice: {val_dice:.4f}', end='')
        if val_dice > best_dice:
            best_dice = val_dice
            torch.save(model.state_dict(), save_path)
            print('  <- Best!', end='')
        print()

    model.load_state_dict(torch.load(save_path, weights_only=True))
    print(f'최고 Val Dice: {best_dice:.4f}')
    return model, history, best_dice


print('train_model() 함수 준비 완료')

In [ ]:
import traceback as _tb
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
os.environ['TQDM_DISABLE'] = '1'   # Optuna trial 중 batch tqdm 비활성화

# === Cell 5: Optuna alpha/gamma 탐색 ===

ALPHA_LOW_PLWCE,  ALPHA_HIGH_PLWCE  = 2.5, 15.0
ALPHA_LOW_PWCE,   ALPHA_HIGH_PWCE   = 0.2,  2.5
PROXY_EPOCHS = 5
PROXY_SUBSET = 0.15
N_TRIALS     = 20
N_TRIALS_PF  = N_TRIALS * 2  # PLWCE+Focal: 파라미터 2개(alpha,gamma)이므로 2배


def make_objective(loss_name, alpha_low, alpha_high):
    def objective(trial):
        alpha = trial.suggest_float('alpha', alpha_low, alpha_high)
        try:
            _, _, dice = train_model(
                loss_name    = loss_name,
                alpha        = alpha,
                epochs       = PROXY_EPOCHS,
                subset_ratio = PROXY_SUBSET,
                tag          = f'trial{trial.number}',
            )
            return dice
        except Exception as e:
            print(f'Trial {trial.number} 실패:'); _tb.print_exc()
            return 0.0
    return objective


# --- PLWCE alpha 탐색 ---
print(f'[Optuna] PLWCE alpha 탐색  (범위: {ALPHA_LOW_PLWCE}~{ALPHA_HIGH_PLWCE}, {N_TRIALS} trials)')
study_plwce = optuna.create_study(
    direction  = 'maximize',
    study_name = 'unet_tn3k_plwce_alpha',
    pruner     = optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=2),
)
study_plwce.optimize(make_objective('plwce_dice', ALPHA_LOW_PLWCE, ALPHA_HIGH_PLWCE), n_trials=N_TRIALS)
best_alpha_plwce = study_plwce.best_params['alpha']
print(f'[PLWCE] 최적 alpha = {best_alpha_plwce:.4f}  (Val Dice = {study_plwce.best_value:.4f})')

# --- PWCE alpha 탐색 ---
print(f'\n[Optuna] PWCE alpha 탐색  (범위: {ALPHA_LOW_PWCE}~{ALPHA_HIGH_PWCE}, {N_TRIALS} trials)')
study_pwce = optuna.create_study(
    direction  = 'maximize',
    study_name = 'unet_tn3k_pwce_alpha',
    pruner     = optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=2),
)
study_pwce.optimize(make_objective('pwce_dice', ALPHA_LOW_PWCE, ALPHA_HIGH_PWCE), n_trials=N_TRIALS)
best_alpha_pwce = study_pwce.best_params['alpha']
print(f'[PWCE]  최적 alpha = {best_alpha_pwce:.4f}  (Val Dice = {study_pwce.best_value:.4f})')

# --- PLWCE+Focal alpha + gamma 공동 탐색 ---
ALPHA_LOW_PF, ALPHA_HIGH_PF = 2.0, 15.0
GAMMA_LOW_PF, GAMMA_HIGH_PF = 0.0,  5.0

print(f'\n[Optuna] PLWCE+Focal alpha+gamma 탐색  '
      f'(alpha: {ALPHA_LOW_PF}~{ALPHA_HIGH_PF}, gamma: {GAMMA_LOW_PF}~{GAMMA_HIGH_PF}, {N_TRIALS} trials)')

def objective_pf(trial):
    alpha = trial.suggest_float('alpha', ALPHA_LOW_PF, ALPHA_HIGH_PF)
    gamma = trial.suggest_float('gamma', GAMMA_LOW_PF, GAMMA_HIGH_PF)
    try:
        _, _, dice = train_model(
            loss_name    = 'plwce_focal_dice',
            alpha        = alpha,
            gamma        = gamma,
            epochs       = PROXY_EPOCHS,
            subset_ratio = PROXY_SUBSET,
            tag          = f'trial{trial.number}',
        )
        return dice
    except Exception as e:
        print(f'Trial {trial.number} 실패:'); _tb.print_exc()
        return 0.0

study_pf = optuna.create_study(
    direction  = 'maximize',
    study_name = f'unet_{DOMAIN}_plwce_focal_alpha_gamma',
    pruner     = optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=2),
)
study_pf.optimize(objective_pf, n_trials=N_TRIALS_PF)
best_alpha_pf = study_pf.best_params['alpha']
best_gamma_pf = study_pf.best_params['gamma']
print(f'[PLWCE+Focal] 최적 alpha={best_alpha_pf:.4f}, gamma={best_gamma_pf:.4f}  '
      f'(Val Dice = {study_pf.best_value:.4f})')

# --- Optuna 결과 저장 ---
optuna_results = {
    'plwce': {
        'best_alpha': best_alpha_plwce,
        'best_proxy_dice': study_plwce.best_value,
        'trials': [{'number': t.number, 'alpha': t.params.get('alpha'), 'value': t.value}
                   for t in study_plwce.trials if t.value is not None],
    },
    'pwce': {
        'best_alpha': best_alpha_pwce,
        'best_proxy_dice': study_pwce.best_value,
        'trials': [{'number': t.number, 'alpha': t.params.get('alpha'), 'value': t.value}
                   for t in study_pwce.trials if t.value is not None],
    },
    'plwce_focal': {
        'best_alpha': best_alpha_pf,
        'best_gamma': best_gamma_pf,
        'best_proxy_dice': study_pf.best_value,
        'trials': [{'number': t.number, 'alpha': t.params.get('alpha'),
                    'gamma': t.params.get('gamma'), 'value': t.value}
                   for t in study_pf.trials if t.value is not None],
    },
}
with open(os.path.join(RESULTS_DIR, f'{DOMAIN}_optuna_results.json'), 'w') as f:
    json.dump(optuna_results, f, indent=2, ensure_ascii=False)
print(f'Optuna 결과 저장: {RESULTS_DIR}/{DOMAIN}_optuna_results.json')

# --- 탐색 결과 시각화 ---
fig, axes = plt.subplots(1, 3, figsize=(21, 5))
for ax, study, sname, a_range in [
    (axes[0], study_plwce, 'PLWCE', f'{ALPHA_LOW_PLWCE}~{ALPHA_HIGH_PLWCE}'),
    (axes[1], study_pwce,  'PWCE',  f'{ALPHA_LOW_PWCE}~{ALPHA_HIGH_PWCE}'),
]:
    trials = [t for t in study.trials if t.value is not None]
    alphas = [t.params['alpha'] for t in trials]
    values = [t.value for t in trials]
    best_a = study.best_params['alpha']
    best_v = study.best_value

    ax.scatter(alphas, values, alpha=0.5, s=40, label='Trials')
    ax.axvline(best_a, color='red', linestyle='--', label=f'Best alpha={best_a:.2f}')
    ax.scatter([best_a], [best_v], color='red', s=100, zorder=5)
    ax.set_xlabel('alpha'); ax.set_ylabel('Val Dice (proxy)')
    ax.set_title(f'{sname} alpha 탐색 (범위 {a_range})')
    ax.legend(); ax.grid(True)

# PLWCE+Focal: 2D scatter (alpha vs gamma, color=Dice)
pf_trials = [t for t in study_pf.trials if t.value is not None]
pf_alphas = [t.params['alpha'] for t in pf_trials]
pf_gammas = [t.params['gamma'] for t in pf_trials]
pf_values = [t.value for t in pf_trials]
sc = axes[2].scatter(pf_alphas, pf_gammas, c=pf_values, cmap='viridis', alpha=0.7, s=60)
axes[2].scatter([best_alpha_pf], [best_gamma_pf], color='red', s=150, zorder=5,
                marker='*', label=f'Best α={best_alpha_pf:.2f}, γ={best_gamma_pf:.2f}')
plt.colorbar(sc, ax=axes[2], label='Val Dice')
axes[2].set_xlabel('alpha'); axes[2].set_ylabel('gamma')
axes[2].set_title(f'PLWCE+Focal alpha+gamma 탐색')
axes[2].legend(fontsize=8); axes[2].grid(True)

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, f'{DOMAIN}_optuna_search.png'), dpi=100)
plt.show()

# PLWCE+Focal: alpha vs gamma 2D 탐색 결과 시각화
fig_pf, ax_pf = plt.subplots(1, 1, figsize=(7, 5))
pf_trials = [t for t in study_pf.trials if t.value is not None]
pf_alphas = [t.params['alpha'] for t in pf_trials]
pf_gammas = [t.params['gamma'] for t in pf_trials]
pf_values = [t.value for t in pf_trials]
sc = ax_pf.scatter(pf_alphas, pf_gammas, c=pf_values, cmap='viridis', alpha=0.7, s=60)
ax_pf.scatter([best_alpha_pf], [best_gamma_pf], color='red', s=150, zorder=5,
              marker='*', label=f'Best α={best_alpha_pf:.2f}, γ={best_gamma_pf:.2f}')
plt.colorbar(sc, ax=ax_pf, label='Val Dice')
ax_pf.set_xlabel('alpha'); ax_pf.set_ylabel('gamma')
ax_pf.set_title('PLWCE+Focal alpha+gamma 탐색')
ax_pf.legend(fontsize=8); ax_pf.grid(True)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, f'{DOMAIN}_optuna_search_pf.png'), dpi=100)
plt.show()
print(f'PLWCE+Focal 탐색 결과 저장: {RESULTS_DIR}/{DOMAIN}_optuna_search_pf.png')
print(f'탐색 결과 저장: {RESULTS_DIR}/{DOMAIN}_optuna_search.png')

os.environ.pop('TQDM_DISABLE', None)   # tqdm 복원

In [ ]:
# === Cell 6: 전체 Loss 비교 학습 ===

FINAL_EPOCHS = 50
FINAL_LR     = 1e-4

# --- Optuna 결과 로드 (Cell 5 미실행 시 JSON fallback) ---
try:
    _ = best_alpha_plwce
except NameError:
    try:
        with open(os.path.join(RESULTS_DIR, f'{DOMAIN}_optuna_results.json')) as f:
            d = json.load(f)
        best_alpha_plwce = d['plwce']['best_alpha']
        best_alpha_pwce  = d['pwce']['best_alpha']
        best_alpha_pf    = d.get('plwce_focal', {}).get('best_alpha', 7.0)
        best_gamma_pf    = d.get('plwce_focal', {}).get('best_gamma', 2.0)
        print(f'Optuna 결과 로드: PLWCE alpha={best_alpha_plwce:.4f}, PWCE alpha={best_alpha_pwce:.4f}, '
              f'PF alpha={best_alpha_pf:.4f} gamma={best_gamma_pf:.4f}')
    except FileNotFoundError:
        best_alpha_plwce = 7.0
        best_alpha_pwce  = 0.5
        best_alpha_pf    = 7.0
        best_gamma_pf    = 2.0
        print('Optuna 미실행 → 기본값 사용 (PLWCE alpha=7.0, PWCE alpha=0.5, PF alpha=7.0 gamma=2.0)')

experiments = [
    ('ce_dice',          1.0,              2.0,             'CE+Dice              (기준선)'),
    ('wce_dice',         1.0,              2.0,             'WCE+Dice'),
    ('lwce_dice',        1.0,              2.0,             'LWCE+Dice'),
    ('plwce_dice',       best_alpha_plwce, 2.0,             f'PLWCE+Dice           (alpha={best_alpha_plwce:.2f})'),
    ('cb_dice',          1.0,              2.0,             'CB+Dice'),
    ('plwce_focal_dice', best_alpha_pf,    best_gamma_pf,   f'PLWCE+Focal+Dice     (α={best_alpha_pf:.2f}, γ={best_gamma_pf:.2f})'),
]

all_results = {}
for loss_name, alpha, gamma, label in experiments:
    model, history, best_dice = train_model(
        loss_name = loss_name,
        alpha     = alpha,
        gamma     = gamma,
        epochs    = FINAL_EPOCHS,
        lr        = FINAL_LR,
        tag       = 'final',
    )
    all_results[label] = {
        'model':     model,
        'history':   history,
        'best_dice': best_dice,
        'loss_name': loss_name,
        'alpha':     alpha,
        'gamma':     gamma,
    }

print('\n' + '='*50)
print('[Loss 비교 실험 요약 — Val Dice]')
print(f"{'Loss':<35} {'Best Val Dice':>13}")
print('-' * 50)
for label, v in all_results.items():
    print(f"{label:<35} {v['best_dice']:>13.4f}")

In [ ]:
# === Cell 7: 평가 및 결과 저장 (시각화) ===

COLORS = ['#4878D0', '#EE854A', '#6ACC65', '#D65F5F', '#B47CC7']

# --- 7-1. 학습 곡선 ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
for i, (label, v) in enumerate(all_results.items()):
    h = v['history']
    ax1.plot(h['loss'],     label=label, color=COLORS[i % len(COLORS)])
    ax2.plot(h['val_dice'], label=label, color=COLORS[i % len(COLORS)])

ax1.set_title('Train Loss'); ax1.set_xlabel('Epoch')
ax1.legend(fontsize=7); ax1.grid(True)
ax2.set_title('Val Dice (Nodule)'); ax2.set_xlabel('Epoch')
ax2.legend(fontsize=7); ax2.grid(True)

plt.suptitle('TN3K — U-Net(ResNet34) 학습 곡선 비교', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, f'{DOMAIN}_training_curves.png'), dpi=100, bbox_inches='tight')
plt.show()
print(f'학습 곡선 저장: {RESULTS_DIR}/{DOMAIN}_training_curves.png')

# --- 7-2. 예측 결과 시각화 (4열: Input / GT / Prob Map / Pred) ---
best_label = max(all_results, key=lambda k: all_results[k]['best_dice'])
best_model = all_results[best_label]['model']
best_model.eval()
print(f'\n시각화 모델: {best_label}  (Val Dice={all_results[best_label]["best_dice"]:.4f})')

val_ds      = TN3KDataset(val_imgs, val_masks, transform=val_tf)
vis_indices = random.sample(range(len(val_ds)), min(4, len(val_ds)))

fig, axes = plt.subplots(len(vis_indices), 4, figsize=(18, len(vis_indices) * 4))
if len(vis_indices) == 1:
    axes = axes[np.newaxis, :]

for row, idx in enumerate(vis_indices):
    img_t, mask_t = val_ds[idx]
    # 역정규화 후 시각화
    img_vis = img_t.numpy().transpose(1, 2, 0) * STD + MEAN
    img_vis = np.clip(img_vis, 0, 1)

    with torch.no_grad():
        logit = best_model(img_t.unsqueeze(0).to(device))  # (1, 1, H, W)
        prob  = torch.sigmoid(logit[0, 0]).cpu().numpy()
        pred  = (prob > 0.5).astype(np.uint8)

    axes[row, 0].imshow(img_vis)
    axes[row, 0].set_title('Input Ultrasound'); axes[row, 0].axis('off')
    axes[row, 1].imshow(mask_t.numpy(), cmap='gray', vmin=0, vmax=1)
    axes[row, 1].set_title('Ground Truth (Nodule=White)'); axes[row, 1].axis('off')
    axes[row, 2].imshow(prob, cmap='jet', vmin=0, vmax=1)
    axes[row, 2].set_title('Nodule Probability Map'); axes[row, 2].axis('off')
    axes[row, 3].imshow(pred, cmap='gray', vmin=0, vmax=1)
    axes[row, 3].set_title(f'Prediction ({best_label.split("(")[0].strip()[:12]})')
    axes[row, 3].axis('off')

plt.suptitle(f'TN3K — 예측 결과 시각화 ({best_label})', fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, f'{DOMAIN}_prediction_vis.png'), dpi=100, bbox_inches='tight')
plt.show()
print(f'예측 결과 저장: {RESULTS_DIR}/{DOMAIN}_prediction_vis.png')

In [ ]:
# === Cell 8: 최종 정량 평가 + JSON + Excel 저장 ===
# 공식 TN3K test set (tn3k/test-image + tn3k/test-mask) 사용
# test_masks가 없을 경우 val_loader로 fallback

if len(test_imgs) > 0 and len(test_masks) > 0:
    eval_loader = test_loader
    eval_set_name = 'Test Set'
    eval_count = len(test_imgs)
    print(f'[최종 평가] 공식 Test Set 사용 ({eval_count}장)')
else:
    eval_loader = val_loader
    eval_set_name = 'Val Set (test mask 없음)'
    eval_count = len(val_imgs)
    print(f'[최종 평가] Test mask 없음 → Val Set 사용 ({eval_count}장)')

print(f'\n[전체 모델 종합 평가 — {eval_set_name}]')
print(f"{'Loss':<35} {'Dice':>7} {'Sens':>7} {'Spec':>7} {'AUC':>7}")
print('-' * 65)

final_results = {}
for label, v in all_results.items():
    metrics = compute_val_metrics(v['model'], eval_loader)
    final_results[label] = {
        'loss_name':     v['loss_name'],
        'alpha':         v['alpha'],
        'best_val_dice': v['best_dice'],
        **metrics,
    }
    print(
        f"{label:<35} "
        f"{metrics['Dice']:>7.4f} "
        f"{metrics['Sensitivity']:>7.4f} "
        f"{metrics['Specificity']:>7.4f} "
        f"{metrics['AUC']:>7.4f}"
    )

# --- 바차트 비교 ---
metric_keys  = ['Dice', 'Sensitivity', 'Specificity', 'AUC']
labels_      = list(final_results.keys())

fig, axes = plt.subplots(1, 4, figsize=(22, 5))
for ax, mkey in zip(axes, metric_keys):
    scores = [final_results[lb][mkey] for lb in labels_]
    bars   = ax.bar(range(len(labels_)), scores, color=COLORS[:len(labels_)], alpha=0.85)
    ax.set_xticks(range(len(labels_)))
    ax.set_xticklabels(
        [lb.split('(')[0].strip()[:12] for lb in labels_],
        rotation=30, ha='right', fontsize=8
    )
    ax.set_title(mkey); ax.set_ylim(0, 1.05); ax.grid(axis='y', alpha=0.4)
    best_idx = int(np.argmax(scores))
    bars[best_idx].set_edgecolor('red'); bars[best_idx].set_linewidth(2.5)
    for bar, score in zip(bars, scores):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
                f'{score:.3f}', ha='center', va='bottom', fontsize=7)

plt.suptitle(f'TN3K — Loss별 최종 평가 지표 비교 ({eval_set_name})', fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, f'{DOMAIN}_final_metrics.png'), dpi=100, bbox_inches='tight')
plt.show()
print(f'평가 차트 저장: {RESULTS_DIR}/{DOMAIN}_final_metrics.png')

# --- JSON 저장 ---
save_data = {
    'domain':         'TN3K Thyroid Nodule Ultrasound Segmentation',
    'model':          'U-Net (ResNet34, ImageNet pretrained)',
    'sota_ref':       {'MRDB (2024)': {'Dice': 0.9002, 'IoU': 0.8185}},
    'num_classes':    NUM_CLASSES,
    'class_counts':   {n: int(c) for n, c in zip(CLASS_NAMES, class_counts)},
    'imbalance':      {'BG_Nodule': round(class_counts[0] / class_counts[1], 1)},
    'train_count':    len(tr_imgs),
    'val_count':      len(val_imgs),
    'test_count':     len(test_imgs),
    'eval_set':       eval_set_name,
    'final_epochs':   FINAL_EPOCHS,
    'results': {
        k: {mk: float(mv) if isinstance(mv, (float, np.floating)) else mv
            for mk, mv in v.items() if mk != 'model'}
        for k, v in final_results.items()
    },
    'best_model': max(final_results, key=lambda k: final_results[k]['Dice']),
}
with open(os.path.join(RESULTS_DIR, f'{DOMAIN}_final_results.json'), 'w', encoding='utf-8') as f:
    json.dump(save_data, f, indent=2, ensure_ascii=False)
print(f'JSON 저장: {RESULTS_DIR}/{DOMAIN}_final_results.json')

# --- Excel 저장 (Summary + Training_History) ---
summary_rows = []
for label, v in final_results.items():
    summary_rows.append({
        'Loss_Function':    label,
        'loss_name':        v['loss_name'],
        'alpha':            round(float(v['alpha']), 4),
        'Best_Val_Dice':    round(v['best_val_dice'], 4),
        'Test_Dice':        round(v['Dice'],        4),
        'Test_Sensitivity': round(v['Sensitivity'], 4),
        'Test_Specificity': round(v['Specificity'], 4),
        'Test_AUC':         round(v['AUC'],         4),
        'eval_set':         eval_set_name,
        'BG_Nodule_ratio':  round(class_counts[0] / class_counts[1], 1),
        'epochs':           FINAL_EPOCHS,
        'model':            'U-Net (ResNet34)',
    })
df_summary = pd.DataFrame(summary_rows)

history_rows = []
for label, v in all_results.items():
    for ep, (loss, dice) in enumerate(
        zip(v['history']['loss'], v['history']['val_dice']), 1
    ):
        history_rows.append({
            'Loss_Function': label,
            'Epoch':         ep,
            'Train_Loss':    round(loss, 6),
            'Val_Dice':      round(dice, 6),
        })
df_history = pd.DataFrame(history_rows)

excel_path = os.path.join(RESULTS_DIR, f'{DOMAIN}_final_results.xlsx')
with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
    df_summary.to_excel(writer, sheet_name='Summary',          index=False)
    df_history.to_excel(writer, sheet_name='Training_History', index=False)
print(f'Excel 저장: {excel_path}')

print(f"\n최고 모델: {save_data['best_model']}")
print(f'BG : Nodule = {save_data["imbalance"]["BG_Nodule"]:>5.1f} : 1')
print(f'평가 기준: {eval_set_name}  ({eval_count}장)')